# Bank Marketing: Data Understanding

## Project objective

The goal of this project is to build and evaluate machine learning models that predict whether a bank customer will subscribe to a term deposit following a marketing campaign.

This is a binary classification problem:

- `yes`: the customer subscribes
- `no`: the customer does not subscribe

The project will follow the complete machine learning lifecycle, including data understanding, exploratory data analysis, preprocessing, feature engineering, model comparison, hyperparameter tuning, evaluation, and critical reflection.

In [1]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
Python executable: E:\PycharmProjects\MLProjectResit\.venv\Scripts\python.exe


## Dataset loading

The Bank Marketing dataset is retrieved from the UCI Machine Learning Repository.

The dataset contains information from direct marketing campaigns conducted by a Portuguese banking institution. The objective is to predict whether a client subscribed to a term deposit.

In [2]:
from ucimlrepo import fetch_ucirepo

bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (45211, 16)
Target shape: (45211, 1)


## Initial dataset inspection

Before cleaning or transforming the data, the dataset structure is inspected to understand:

- the available features
- the target variable
- the data types
- a small sample of the observations

No changes are made to the dataset at this stage.

In [3]:
print("Feature columns:")
print(X.columns.tolist())

print("\nTarget columns:")
print(y.columns.tolist())

print("\nFeature data types:")
print(X.dtypes)

print("\nFirst five feature rows:")
display(X.head())

print("\nFirst five target rows:")
display(y.head())

Feature columns:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day_of_week', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']

Target columns:
['y']

Feature data types:
age             int64
job            object
marital        object
education      object
default        object
balance         int64
housing        object
loan           object
contact        object
day_of_week     int64
month          object
duration        int64
campaign        int64
pdays           int64
previous        int64
poutcome       object
dtype: object

First five feature rows:


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN



First five target rows:


,y
0,no
1,no
2,no
3,no
4,no


## Combining features and target

For initial exploration, the feature matrix and target column are combined into a single dataframe.

The original `X` and `y` objects are preserved separately so they can later be used safely during preprocessing and model development.

In [4]:
import pandas as pd

df = pd.concat([X, y], axis=1)

print("Combined dataset shape:", df.shape)

display(df.head())

df.info()

Combined dataset shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
 16  y            45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


## Missing-value overview

The number and percentage of missing values are calculated for every column.

At this stage, missing values are only identified and measured. No values are removed or imputed yet.

In [5]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values(
    by="missing_percentage",
    ascending=False
)

display(missing_summary)

,missing_count,missing_percentage
poutcome,36959,81.75
contact,13020,28.80
education,1857,4.11
job,288,0.64


## Inspection of columns with missing values

The unique values and their frequencies are inspected for columns containing missing data.

This helps determine whether missing values represent genuinely unavailable information, a special category, or a data-quality problem.

In [6]:
columns_with_missing = missing_summary.index.tolist()

for column in columns_with_missing:
    print(f"\nColumn: {column}")
    print(df[column].value_counts(dropna=False))


Column: poutcome
poutcome
NaN        36959
failure     4901
other       1840
success     1511
Name: count, dtype: int64

Column: contact
contact
cellular     29285
NaN          13020
telephone     2906
Name: count, dtype: int64

Column: education
education
secondary    23202
tertiary     13301
primary       6851
NaN           1857
Name: count, dtype: int64

Column: job
job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
NaN               288
Name: count, dtype: int64


## Duplicate-row inspection

Duplicate observations can distort descriptive statistics and model evaluation if identical records appear in both the training and test data.

The dataset is checked for fully duplicated rows before any preprocessing is applied.

In [7]:
duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print("Number of fully duplicated rows:", duplicate_count)
print(f"Percentage of duplicated rows: {duplicate_percentage:.2f}%")

Number of fully duplicated rows: 0
Percentage of duplicated rows: 0.00%


## Target-variable distribution

The distribution of the target variable is inspected to determine whether the classification problem is balanced.

A strong imbalance would make accuracy less informative and may require metrics such as precision, recall, F1-score, ROC-AUC, and PR-AUC.

In [8]:
target_counts = df["y"].value_counts()
target_percentages = (df["y"].value_counts(normalize=True) * 100).round(2)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

display(target_summary)

,count,percentage
y,,
no,39922,88.3
yes,5289,11.7


### Target-distribution finding

The target variable is imbalanced. Approximately 88.3% of customers did not subscribe, while only 11.7% subscribed.

Because of this imbalance, accuracy alone would be misleading. A naive model predicting `no` for every customer would achieve high accuracy without identifying any actual subscribers.

Therefore, later evaluation will focus on metrics such as:

- recall
- precision
- F1-score
- ROC-AUC
- PR-AUC
- confusion matrix

## Feature-type identification

The input features are separated into numerical and categorical groups.

This distinction is necessary because the two groups will require different preprocessing steps later in the machine learning pipeline.

In [9]:
numerical_columns = X.select_dtypes(include="number").columns.tolist()
categorical_columns = X.select_dtypes(exclude="number").columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

print("\nNumber of numerical columns:", len(numerical_columns))
print("Number of categorical columns:", len(categorical_columns))

Numerical columns:
['age', 'balance', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous']

Categorical columns:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Number of numerical columns: 7
Number of categorical columns: 9


## Numerical feature summary

Descriptive statistics are calculated for the numerical features to understand their central tendency, spread, range, and possible extreme values.

These statistics provide an initial indication of skewness and potential outliers.

In [10]:
numerical_summary = df[numerical_columns].describe().T

numerical_summary["median"] = df[numerical_columns].median()
numerical_summary["skewness"] = df[numerical_columns].skew()

display(numerical_summary)

,count,mean,std,min,25%,50%,75%,max,median,skewness
age,45211.0,40.936210,10.618762,18.0,33.0,39.0,48.0,95.0,39.0,0.684818
balance,45211.0,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0,448.0,8.360308
day_of_week,45211.0,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0,16.0,0.093079
duration,45211.0,258.163080,257.527812,0.0,103.0,180.0,319.0,4918.0,180.0,3.144318
campaign,45211.0,2.763841,3.098021,1.0,1.0,2.0,3.0,63.0,2.0,4.898650
pdays,45211.0,40.197828,100.128746,-1.0,-1.0,-1.0,-1.0,871.0,-1.0,2.615715
previous,45211.0,0.580323,2.303441,0.0,0.0,0.0,0.0,275.0,0.0,41.846454


### Numerical-summary findings

Several numerical features are strongly right-skewed.

- `balance` contains negative values and a small number of very large positive values.
- `duration` and `campaign` contain possible extreme observations.
- `previous` is highly skewed because most observations are zero.
- `pdays` has a median and lower quartiles of `-1`, indicating that `-1` is likely a special sentinel value representing no previous contact rather than an actual number of days.

These findings will be investigated further before deciding whether scaling, transformation, or special-value handling is required.

## Categorical feature summary

The number of unique values and the most frequent category are inspected for each categorical feature.

This helps identify high-cardinality features, dominant categories, and unusual encoded values.

In [11]:
categorical_summary = pd.DataFrame({
    "unique_values": df[categorical_columns].nunique(dropna=False),
    "most_frequent": df[categorical_columns].mode(dropna=False).iloc[0],
    "most_frequent_count": [
        df[column].value_counts(dropna=False).iloc[0]
        for column in categorical_columns
    ]
})

categorical_summary["most_frequent_percentage"] = (
    categorical_summary["most_frequent_count"] / len(df) * 100
).round(2)

display(categorical_summary)

,unique_values,most_frequent,most_frequent_count,most_frequent_percentage
job,12,blue-collar,9732,21.53
marital,3,married,27214,60.19
education,4,secondary,23202,51.32
default,2,no,44396,98.20
housing,2,yes,25130,55.58
loan,2,no,37967,83.98
contact,3,cellular,29285,64.77
month,12,may,13766,30.45
poutcome,4,NaN,36959,81.75


## Data-understanding conclusion

The dataset contains 45,211 observations, 16 input features, and one binary target variable.

The initial inspection identified several important characteristics:

- The target is imbalanced, with only 11.7% of customers subscribing.
- Missing values occur only in categorical features, particularly `poutcome` and `contact`.
- No fully duplicated rows were found.
- Several numerical features are strongly right-skewed and contain possible extreme values.
- `pdays = -1` appears to represent customers who were not previously contacted.
- The dataset contains both numerical and categorical variables and will require separate preprocessing strategies.
- The `duration` feature requires special consideration because it is only known after a marketing call has occurred and may cause leakage in a pre-contact prediction model.

These findings will guide the exploratory analysis, preprocessing pipeline, model evaluation strategy, and leakage controls.